<a href="https://colab.research.google.com/github/BerkeleyExpertSystemTechnologiesLab/Squishy-Methane-Analysis/blob/jberry/Squish_Robot_Quant_Model_v5_1_mm_%2B_image_transforms.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Model Description

This model was produced by and for Squishy Robotics for the task of identifying and classifying methane leaks.


This model was made in conjunction with a synthetic dataset of 1 channel, 240 by 320 greyscale images of methane leaks
(1 x 240 x 320)


This model is experimental and uses the Optuna Hyperparameter Optimizer to search for successful hyperparameters (Learning Rate, Optimizer, Batch Size, Dropout %, etc...) and different optimizers. As such if you want to test a specific Model architecture you need to comment out the Optuna code and run a train/test on that specific model.

In [1]:
pip install optuna #Hyperparameter Optimizer

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 413.9/413.9 kB 10.6 MB/s eta 0:00:00


In [2]:
import os
import numpy as np

from collections import defaultdict
from collections import Counter

import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
import torchvision

from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
from torchvision.models import swin_t, SwinTransformer

from sklearn.model_selection import train_test_split
from torch.utils.data import random_split

# Hyperparameter Search
import optuna

import json
import glob

#For file uploading
from google.colab import files
from google.colab import drive
from google.colab import auth


In [3]:
#Download the file
auth.authenticate_user()
drive.mount('/content/drive')

Mounted at /content/drive


In [4]:
# This may take several minutes, the synthetic dataset can be large
!unzip -q "/content/drive/MyDrive/Squishy_Robotics_Dataset/Final_Dataset_single_channel_w_artif.zip" -d /content/

## Print out the shape of the data

In [6]:
# Loading an example file to demonstrate the dimensions
# This file might not exist, change the name to one that does to show the
# dimensions
file_path = './Final_Dataset_single_channel_w_artif/data/class_0/1237_frame_02_class_0.npy'
sample_data = np.load(file_path)
print(f"Shape of preprocessed sample data: {sample_data.shape}")
print(f"Data type of preprocessed sample data: {sample_data.dtype}")

# GasVid synthetic processed dataset should be 2 channels, 240x320 in dimension

Shape of preprocessed sample data: (1, 240, 320)
Data type of preprocessed sample data: float32


In [7]:
# Assuming the data is in 'Final_Dataset/data' and class folders are named 'class_0' ... 'class_7'
data_dir = 'Final_Dataset_single_channel_w_artif/data'
classes = sorted(os.listdir(data_dir))
print(f"Classes: {classes}")

Classes: ['class_0', 'class_1', 'class_2', 'class_3', 'class_4', 'class_5', 'class_6', 'class_7']


## Create a dataset and dataloader

In [8]:
class Multi_Modal_Dataset(Dataset):
    def __init__(self, numpy_files, json_files, labels, transform=None):
        """
        numpy_dir points to all the numpy 2 channel frames that were collected
          from METEC. This is designed to be 1st Channel Greyscale image of
          background, 2nd channel is just the gas plume scaled to some ppm
        json_dir points to all the metadata (ppm, distance, etc) that was
          collected from METEC or estimated using BEST Labs algorithms
        """
        self.numpy_files = numpy_files
        self.json_files = json_files
        self.labels = labels
        self.transform = transform


    def __len__(self):
      return len(self.numpy_files)


    def __getitem__(self, idx):
      numpy_path = self.numpy_files[idx]
      image_data = np.load(numpy_path)
      image_tensor = torch.from_numpy(image_data).float()

      if self.transform:
        image_tensor = self.transform(image_tensor)

      json_path = self.json_files[idx]
      with open(json_path, 'r') as f:
        metadata = json.load(f)

      metadat_features = self._extract_metadata_features(metadata)
      metadata_tensor = torch.tensor(metadat_features, dtype=torch.float32)

      label = self.labels[idx]

      return image_tensor, metadata_tensor, label


    def _extract_metadata_features(self, metadata):
      """
      Extracts a few entries from the metadata.
        For now:
          distance
          ppm
        In the future
          windspeed
          angle?
      """

      features = []

      # If the features exist, extract them, else place 0.0
      # Print warning statements if unable to retrieve the data
      distance = metadata.get("distance_m", None)
      if distance is None or distance == 0.0:
          print(f"WARNING: Invalid or missing distance_m value: {distance}")
          print(f"  Metadata keys available: {list(metadata.keys())}")
          features.append(0.0)
      else:
          features.append(distance)

      ppm = metadata.get("ppm", None)
      if ppm is None:
          print(f"WARNING: Missing ppm value")
          print(f"  Metadata keys available: {list(metadata.keys())}")
          features.append(0.0)
      else:
          features.append(ppm)

      return features

In [9]:
numpy_dir = "./Final_Dataset_single_channel_w_artif/data"
json_dir = "./Final_Dataset_single_channel_w_artif/metadata"

all_numpy_files = []
all_json_files = []
all_labels = []

print(f"Looking in: {numpy_dir}")
print(f"Directory exists: {os.path.exists(numpy_dir)}\n")

# Load each class separatley, collect the numpy and json files for a certain
# class at the same time
for class_idx in range(8):
    numpy_class_dir = os.path.join(numpy_dir, f"class_{class_idx}")
    json_class_dir = os.path.join(json_dir, f"class_{class_idx}")

    numpy_files_in_class = sorted(glob.glob(os.path.join(numpy_class_dir, "*.npy")))

    print(f"Class {class_idx}: Found {len(numpy_files_in_class)} files")

    for numpy_file in numpy_files_in_class:
        base_name = os.path.splitext(os.path.basename(numpy_file))[0]
        video_id = base_name.split('_')[0]


        json_filename = f"{video_id}_class_{class_idx}.json"
        json_file = os.path.join(json_class_dir, json_filename)

        if os.path.exists(json_file):
            all_numpy_files.append(numpy_file)
            all_json_files.append(json_file)
            all_labels.append(class_idx)
        else:
            print(f"WARNING: JSON missing for {base_name}")

print(f"\n{'='*60}")
print(f"TOTAL: {len(all_numpy_files)} numpy files")
print(f"TOTAL: {len(all_json_files)} json files")
print(f"{'='*60}\n")

# Only continue if we have files
if len(all_numpy_files) == 0:
    raise ValueError("!!!No files found!!! Check your paths above.")

# Now continue with video splitting
video_to_indices = defaultdict(list)
for idx, numpy_file in enumerate(all_numpy_files):
    video_id = os.path.basename(numpy_file).split('_')[0]
    video_to_indices[video_id].append(idx)

print(f"Number of unique videos: {len(video_to_indices)}")
print(f"Video IDs: {sorted(video_to_indices.keys())}\n")


Looking in: ./Final_Dataset_single_channel_w_artif/data
Directory exists: True

Class 0: Found 5391 files
Class 1: Found 5403 files
Class 2: Found 5410 files
Class 3: Found 5393 files
Class 4: Found 5421 files
Class 5: Found 5412 files
Class 6: Found 5394 files
Class 7: Found 5395 files

TOTAL: 43219 numpy files
TOTAL: 43219 json files

Number of unique videos: 28
Video IDs: ['1237', '1238', '1239', '1240', '1241', '1242', '1467', '1468', '1469', '1470', '1471', '1472', '2559', '2560', '2561', '2562', '2563', '2564', '2566', '2567', '2568', '2569', '2571', '2578', '2579', '2580', '2581', '2583']



In [10]:
video_to_indices = defaultdict(list) #Make an empty dictionary of lists

for idx, numpy_file in enumerate(all_numpy_files):
    video_id = os.path.basename(numpy_file).split('_')[0] #Extract 4 digit code from numpy filename
    video_to_indices[video_id].append(idx)

video_ids = list(video_to_indices.keys())

# Split the video into train and test
train_vids, test_vids = train_test_split(video_ids, test_size=0.2, random_state=42)

# Verify no overlap
overlap = set(train_vids) & set(test_vids)
if overlap:
    print(f"\nVideos overlap: {overlap}")
else:
    print(f"\nNo video overlap - train and test are separate")

train_indices = []
test_indices = []

for vid in train_vids:
    train_indices.extend(video_to_indices[vid])
for vid in test_vids:
    test_indices.extend(video_to_indices[vid])

# Create file lists
train_numpy = [all_numpy_files[i] for i in train_indices]
train_json = [all_json_files[i] for i in train_indices]
train_labels_list = [all_labels[i] for i in train_indices]

test_numpy = [all_numpy_files[i] for i in test_indices]
test_json = [all_json_files[i] for i in test_indices]
test_labels_list = [all_labels[i] for i in test_indices]



No video overlap - train and test are separate


## Image Transformations

In [11]:
# Augmentation section
# https://docs.pytorch.org/vision/0.13/transforms.html
train_transforms = transforms.Compose([
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomVerticalFlip(p=0.5),
    transforms.RandomRotation(degrees=15),
    transforms.RandomAffine(
        degrees=0,
        translate=(0.1, 0.1),
        scale=(0.9, 1.1),
    ),
    transforms.RandomApply([
        transforms.GaussianBlur(
            kernel_size=3,
            sigma=(0.1, 2.0)
        )
    ], p=0.3)
])

#During testing don't use augmentation
test_transforms = None

In [12]:
# SHOW FINAL SPLIT STATISTICS
print(f"\n{'='*60}")
print("DATASET STATISTICS")
print("="*90)

print(f"\nTRAINING SET:")
print(f"   Total samples: {len(train_numpy)}")
print(f"   From {len(train_vids)} videos: {sorted(train_vids)}")

# Count samples per class in training
train_class_counts = Counter(train_labels_list)
print(f"\n   Samples per class:")
for class_id in range(8):
    count = train_class_counts.get(class_id, 0)
    percentage = (count / len(train_numpy) * 100) if len(train_numpy) > 0 else 0
    print(f"      Class {class_id}: {count:5d} samples ({percentage:5.2f}%)")

print(f"\nTEST SET:")
print(f"   Total samples: {len(test_numpy)}")
print(f"   From {len(test_vids)} videos: {sorted(test_vids)}")

# Count samples per class in testing
test_class_counts = Counter(test_labels_list)
print(f"\n   Samples per class:")
for class_id in range(8):
    count = test_class_counts.get(class_id, 0)
    percentage = (count / len(test_numpy) * 100) if len(test_numpy) > 0 else 0
    print(f"      Class {class_id}: {count:5d} samples ({percentage:5.2f}%)")

# VERIFY ALL CLASSES PRESENT
print(f"\n{'='*70}")
print("VERIFICATION")
print("="*70)

train_classes = set(train_labels_list)
test_classes = set(test_labels_list)
missing_train = set(range(8)) - train_classes
missing_test = set(range(8)) - test_classes

if missing_train:
    print(f"WARNING: Training missing classes {missing_train}")
else:
    print(f"Training set has all 8 classes")

if missing_test:
    print(f"WARNING: Testing missing classes {missing_test}")
else:
    print(f"Test set has all 8 classes")

# Show train/test split ratio
total_samples = len(train_numpy) + len(test_numpy)
train_ratio = len(train_numpy) / total_samples * 100
test_ratio = len(test_numpy) / total_samples * 100
print(f"\nSplit ratio: {train_ratio:.1f}% train / {test_ratio:.1f}% test")

print(f"\n{'='*70}")
print("DATA SPLIT COMPLETE AND VERIFIED")
print("="*70)



DATASET STATISTICS

TRAINING SET:
   Total samples: 33973
   From 22 videos: ['1238', '1239', '1240', '1241', '1242', '1467', '1468', '1471', '1472', '2560', '2561', '2562', '2563', '2564', '2566', '2567', '2568', '2571', '2578', '2579', '2581', '2583']

   Samples per class:
      Class 0:  4236 samples (12.47%)
      Class 1:  4258 samples (12.53%)
      Class 2:  4250 samples (12.51%)
      Class 3:  4239 samples (12.48%)
      Class 4:  4260 samples (12.54%)
      Class 5:  4241 samples (12.48%)
      Class 6:  4241 samples (12.48%)
      Class 7:  4248 samples (12.50%)

TEST SET:
   Total samples: 9246
   From 6 videos: ['1237', '1469', '1470', '2559', '2569', '2580']

   Samples per class:
      Class 0:  1155 samples (12.49%)
      Class 1:  1145 samples (12.38%)
      Class 2:  1160 samples (12.55%)
      Class 3:  1154 samples (12.48%)
      Class 4:  1161 samples (12.56%)
      Class 5:  1171 samples (12.66%)
      Class 6:  1153 samples (12.47%)
      Class 7:  1147 samples

In [13]:
train_dataset = Multi_Modal_Dataset(train_numpy,
                                    train_json,
                                    train_labels_list,
                                    transform=train_transforms)
test_dataset = Multi_Modal_Dataset(test_numpy,
                                   test_json,
                                   test_labels_list,
                                   transform=test_transforms)

# Define Swin ViT model
This model uses the pytorch Swin ViT default:

https://docs.pytorch.org/vision/main/models/swin_transformer.html

## Define the Optuna Objective Function

This function will be called by Optuna for each trial. It will:
1. Suggest hyperparameters using the trial object.
2. Build and train the Swin ViT model with the suggested hyperparameters.
3. Evaluate the model on a validation set
4. Return the metric to minimize (loss) or maximize (accuracy).

In [14]:
def build_swin_backbone(in_channels=1):
    model = swin_t(weights=None, num_classes=8)  # num_classes ignored after we replace head
    # First layer: patch embedding. Default is Conv2d(3, 96, ...). Use 1 channel.
    old = model.features[0][0]
    model.features[0][0] = nn.Conv2d(
        in_channels,
        old.out_channels,
        kernel_size=old.kernel_size,
        stride=old.stride,
        padding=old.padding,
    )
    # Output features instead of logits (swin_t last stage dim = 96 * 2^3 = 768)
    model.head = nn.Identity()
    return model, 768

In [15]:
def objective(trial):

    #############################
    # All Hyperparameters Tested
    #############################
    lr = trial.suggest_float('lr', 1e-5, 1e-1, log=True)
    optimizer_name = trial.suggest_categorical('optimizer', ['Adam', 'SGD', 'AdamW'])
    momentum = trial.suggest_float('momentum', 0.0, 0.99) if optimizer_name in ['SGD'] else 0.0
    weight_decay = trial.suggest_float('weight_decay', 0.0, 0.01)
    batch_size = trial.suggest_categorical('batch_size', [16, 32, 64, 128])
    num_epochs = trial.suggest_int('num_epochs', 10, 25)
    fc_drop_rate = trial.suggest_float('fc_drop_rate', 0.2, 0.6)

    #####################
    # Define the Model
    #####################
    class MultiModeSwinViT(nn.Module):
        def __init__(self, num_classes=8, in_channels=1, num_metadata_feats=2, fc_drop_rate=0.3):
            super(MultiModeSwinViT, self).__init__()

            #Swin ViT for images only
            self.swin, self._swin_features = build_swin_backbone(in_channels=in_channels)

            #Smaller Neural Net for metadata only
            self.metadata_fc = nn.Sequential(
                nn.Linear(num_metadata_feats, 64),
                nn.ReLU(),
                nn.Dropout(fc_drop_rate),
                nn.Linear(64,64)
            )

            #Classifier combines metadata NN and image Swin ViT outputs
            self.classifier = nn.Sequential(
                nn.Linear(768 + 64, 128),
                nn.ReLU(),
                nn.Dropout(fc_drop_rate),
                nn.Linear(128, num_classes)
            )

        def forward(self, image, metadata):
            swin_out = self.swin(image)
            meta_out = self.metadata_fc(metadata)
            combined = torch.cat([swin_out, meta_out], dim=1)
            output = self.classifier(combined)

            return output

    model = MultiModeSwinViT(
        num_classes=8,
        in_channels=1,
        num_metadata_feats=2,
        fc_drop_rate=fc_drop_rate
    )

    ###############################
    # Define optimizer
    ###############################
    if optimizer_name == 'SGD':
        optimizer = optim.SGD(model.parameters(), lr=lr, momentum=momentum, weight_decay=weight_decay)
    elif optimizer_name == 'Adam':
        optimizer = optim.Adam(model.parameters(), lr=lr, weight_decay=weight_decay)
    elif optimizer_name == 'AdamW':
        optimizer = optim.AdamW(model.parameters(), lr=lr, weight_decay=weight_decay)
    else:
        raise ValueError(f"Unknown optimizer name: {optimizer_name}")

    criterion = nn.CrossEntropyLoss()

    ##########################################
    # Create DataLoaders with trial batch_size
    ##########################################
    train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
    test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False)

    ###############################
    # Train the model
    ###############################
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    model.to(device)


    print(f"\n{'='*70}")
    print(f"Trial {trial.number} | lr={lr:.6f} | optimizer={optimizer_name} | "
          f"batch={batch_size} | epochs={num_epochs} | fc_drop={fc_drop_rate:.3f}")
    print(f"{'='*70}")


    model.train()
    train_correct = 0
    train_total = 0
    for epoch in range(num_epochs):
        train_correct = 0
        train_total = 0
        train_loss = 0.0
        num_batches = 0

        for images, metadata, labels in train_loader:
            images = images.to(device)
            metadata = metadata.to(device)
            labels = labels.to(device)

            optimizer.zero_grad()
            outputs = model(images, metadata)
            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()

            # Calculate loss
            train_loss += loss.item()
            num_batches += 1

            # Calculate training accuracy
            _, predicted = torch.max(outputs.data, 1)
            train_total += labels.size(0)
            train_correct += (predicted == labels).sum().item()

        #Print out training during each epoch
        train_accuracy = train_correct / train_total
        avg_train_loss = train_loss / num_batches

        # Print training accuracy for this epoch
        print(f"Epoch [{epoch+1:2d}/{num_epochs}] Train Loss: {avg_train_loss:.4f} | Train Acc: {train_accuracy:.4f}")

    #####################
    # Evaluate the model
    #####################
    model.eval()
    correct, total = 0, 0
    val_loss = 0.0
    num_val_batches = 0
    with torch.no_grad():

        for images, metadata, labels in test_loader:
            images = images.to(device)
            metadata = metadata.to(device)
            labels = labels.to(device)

            # Calculate validation loss
            outputs = model(images, metadata)
            loss = criterion(outputs, labels)
            val_loss += loss.item()
            num_val_batches += 1

            # Calculate Validation Accuract
            _, predicted = torch.max(outputs.data, 1)
            total += labels.size(0)
            correct += (predicted == labels).sum().item()

    accuracy = correct / total
    avg_val_loss = val_loss / num_val_batches

    print(f"Validation Loss: {avg_val_loss:.4f} | Validation Acc: {accuracy:.4f}")
    print(f"{'='*70}\n")

    return accuracy


## Run the Optuna Study

Create an Optuna study and run the optimization process.

In [ ]:
# Create a study object and specify the direction of optimization (maximize accuracy)
study = optuna.create_study(direction='maximize',
                             pruner=optuna.pruners.MedianPruner(n_startup_trials=5, n_warmup_steps=5))

# Run the optimization
study.optimize(objective, n_trials = 10)

# Print the best hyperparameters found
print("Best hyperparameters: ", study.best_params)

# Print the best accuracy found
print("Best accuracy: ", study.best_value)

# Plot the visualization
optuna.visualization.plot_param_importances(study).show()

# Run more trials
# study.optimize(objective, n_trials=20)

[I 2026-02-05 01:02:03,608] A new study created in memory with name: no-name-1b2a0154-711d-4144-b521-5186fa4f14ab



Trial 0 | lr=0.006966 | optimizer=Adam | batch=32 | epochs=22 | fc_drop=0.278
Epoch [ 1/22] Train Loss: 1.4028 | Train Acc: 0.4031
Epoch [ 2/22] Train Loss: 1.0959 | Train Acc: 0.5148
Epoch [ 3/22] Train Loss: 1.0531 | Train Acc: 0.5315
Epoch [ 4/22] Train Loss: 1.0222 | Train Acc: 0.5427
Epoch [ 5/22] Train Loss: 0.9976 | Train Acc: 0.5572
Epoch [ 6/22] Train Loss: 0.9862 | Train Acc: 0.5652
Epoch [ 7/22] Train Loss: 0.9617 | Train Acc: 0.5764
Epoch [ 8/22] Train Loss: 0.9527 | Train Acc: 0.5789
Epoch [ 9/22] Train Loss: 0.9568 | Train Acc: 0.5734
Epoch [10/22] Train Loss: 0.9525 | Train Acc: 0.5762
Epoch [11/22] Train Loss: 0.9396 | Train Acc: 0.5853
Epoch [12/22] Train Loss: 0.9463 | Train Acc: 0.5829
Epoch [13/22] Train Loss: 0.9384 | Train Acc: 0.5828
Epoch [14/22] Train Loss: 0.9243 | Train Acc: 0.5917
Epoch [15/22] Train Loss: 0.9193 | Train Acc: 0.5942
Epoch [16/22] Train Loss: 0.9332 | Train Acc: 0.5901
Epoch [17/22] Train Loss: 0.9258 | Train Acc: 0.5920
Epoch [18/22] Train 

[I 2026-02-05 06:55:26,132] Trial 0 finished with value: 0.6660177373999567 and parameters: {'lr': 0.006965638521005632, 'optimizer': 'Adam', 'weight_decay': 0.0019283061568557336, 'batch_size': 32, 'num_epochs': 22, 'fc_drop_rate': 0.2779968407746161}. Best is trial 0 with value: 0.6660177373999567.


Validation Loss: 0.7126 | Validation Acc: 0.6660


Trial 1 | lr=0.000838 | optimizer=SGD | batch=32 | epochs=11 | fc_drop=0.246
Epoch [ 1/11] Train Loss: 1.8161 | Train Acc: 0.2637
Epoch [ 2/11] Train Loss: 1.6097 | Train Acc: 0.3270
Epoch [ 3/11] Train Loss: 1.5004 | Train Acc: 0.3700
Epoch [ 4/11] Train Loss: 1.4253 | Train Acc: 0.3928
Epoch [ 5/11] Train Loss: 1.3766 | Train Acc: 0.4088
Epoch [ 6/11] Train Loss: 1.3376 | Train Acc: 0.4225
Epoch [ 7/11] Train Loss: 1.3127 | Train Acc: 0.4298
Epoch [ 8/11] Train Loss: 1.2839 | Train Acc: 0.4450
Epoch [ 9/11] Train Loss: 1.2661 | Train Acc: 0.4500
Epoch [10/11] Train Loss: 1.2499 | Train Acc: 0.4606
Epoch [11/11] Train Loss: 1.2364 | Train Acc: 0.4659


[I 2026-02-05 09:58:19,057] Trial 1 finished with value: 0.7500540774388925 and parameters: {'lr': 0.0008376128549279948, 'optimizer': 'SGD', 'momentum': 0.5661899162755285, 'weight_decay': 0.00369988461903655, 'batch_size': 32, 'num_epochs': 11, 'fc_drop_rate': 0.24594199709395764}. Best is trial 1 with value: 0.7500540774388925.


Validation Loss: 1.1717 | Validation Acc: 0.7501


Trial 2 | lr=0.042866 | optimizer=SGD | batch=32 | epochs=14 | fc_drop=0.483
Epoch [ 1/14] Train Loss: 1.7620 | Train Acc: 0.2740
Epoch [ 2/14] Train Loss: 1.6392 | Train Acc: 0.3090
Epoch [ 3/14] Train Loss: 1.6026 | Train Acc: 0.3205
Epoch [ 4/14] Train Loss: 1.5971 | Train Acc: 0.3232
Epoch [ 5/14] Train Loss: 1.5916 | Train Acc: 0.3237
Epoch [ 6/14] Train Loss: 1.5903 | Train Acc: 0.3257
Epoch [ 7/14] Train Loss: 1.5769 | Train Acc: 0.3272
Epoch [ 8/14] Train Loss: 1.5767 | Train Acc: 0.3334
Epoch [ 9/14] Train Loss: 1.5736 | Train Acc: 0.3317
Epoch [10/14] Train Loss: 1.5768 | Train Acc: 0.3330
Epoch [11/14] Train Loss: 1.5759 | Train Acc: 0.3305
Epoch [12/14] Train Loss: 1.5796 | Train Acc: 0.3284
Epoch [13/14] Train Loss: 1.5791 | Train Acc: 0.3285
Epoch [14/14] Train Loss: 1.5722 | Train Acc: 0.3321


[I 2026-02-05 13:44:19,671] Trial 2 finished with value: 0.2489725286610426 and parameters: {'lr': 0.04286560134981259, 'optimizer': 'SGD', 'momentum': 0.42761417627019466, 'weight_decay': 0.002061374713701838, 'batch_size': 32, 'num_epochs': 14, 'fc_drop_rate': 0.48302781607469913}. Best is trial 1 with value: 0.7500540774388925.


Validation Loss: 1.6433 | Validation Acc: 0.2490


Trial 3 | lr=0.001633 | optimizer=Adam | batch=32 | epochs=20 | fc_drop=0.230
Epoch [ 1/20] Train Loss: 1.3593 | Train Acc: 0.4145
Epoch [ 2/20] Train Loss: 1.1267 | Train Acc: 0.5057
Epoch [ 3/20] Train Loss: 0.9640 | Train Acc: 0.5717
Epoch [ 4/20] Train Loss: 0.8123 | Train Acc: 0.6422
Epoch [ 5/20] Train Loss: 0.7442 | Train Acc: 0.6749
Epoch [ 6/20] Train Loss: 0.7082 | Train Acc: 0.6923
Epoch [ 7/20] Train Loss: 0.6823 | Train Acc: 0.7066
Epoch [ 8/20] Train Loss: 0.6672 | Train Acc: 0.7196
Epoch [ 9/20] Train Loss: 0.6405 | Train Acc: 0.7313
Epoch [10/20] Train Loss: 0.6260 | Train Acc: 0.7394
Epoch [11/20] Train Loss: 0.6069 | Train Acc: 0.7508
Epoch [12/20] Train Loss: 0.6074 | Train Acc: 0.7488
Epoch [13/20] Train Loss: 0.5789 | Train Acc: 0.7665
Epoch [14/20] Train Loss: 0.5854 | Train Acc: 0.7612
Epoch [15/20] Train Loss: 0.5650 | Train Acc: 0.7741
Epoch [16/20] Train Loss: 0.5599 | Train Acc: 0.7721
Epoch [17/20] Train Los

[I 2026-02-05 19:00:54,472] Trial 3 finished with value: 0.7919100151416829 and parameters: {'lr': 0.001632906252518328, 'optimizer': 'Adam', 'weight_decay': 0.003790543728557697, 'batch_size': 32, 'num_epochs': 20, 'fc_drop_rate': 0.2299261602385556}. Best is trial 3 with value: 0.7919100151416829.


Validation Loss: 0.5173 | Validation Acc: 0.7919


Trial 4 | lr=0.000073 | optimizer=Adam | batch=16 | epochs=21 | fc_drop=0.251
Epoch [ 1/21] Train Loss: 1.7555 | Train Acc: 0.2800
Epoch [ 2/21] Train Loss: 1.4534 | Train Acc: 0.3800
Epoch [ 3/21] Train Loss: 1.3268 | Train Acc: 0.4316
Epoch [ 4/21] Train Loss: 1.2586 | Train Acc: 0.4581
Epoch [ 5/21] Train Loss: 1.2175 | Train Acc: 0.4767
Epoch [ 6/21] Train Loss: 1.1859 | Train Acc: 0.4913
Epoch [ 7/21] Train Loss: 1.1599 | Train Acc: 0.5034
Epoch [ 8/21] Train Loss: 1.1376 | Train Acc: 0.5130
Epoch [ 9/21] Train Loss: 1.1183 | Train Acc: 0.5209
Epoch [10/21] Train Loss: 1.0950 | Train Acc: 0.5346


# Sources:
### Hyperparameter Tuning with Optuna:
https://medium.com/@taeefnajib/hyperparameter-tuning-using-optuna-c46d7b29a3e
https://optuna.org/#code_examples

### Multi-Modal ML Models
https://www.nature.com/articles/s41598-025-14901-4
https://www.reddit.com/r/MachineLearning/comments/nziumg/combining_images_and_other_numeric_features_in_a/
https://pyimagesearch.com/2019/02/04/keras-multiple-inputs-and-mixed-data/

### ViT Models
https://www.geeksforgeeks.org/deep-learning/building-a-vision-transformer-from-scratch-in-pytorch/

https://www.youtube.com/watch?v=7o1jpvapaT0&t=2924s

https://medium.com/correll-lab/building-a-vision-transformer-model-from-scratch-a3054f707cc6
